## Ingest Dimention Data to gold

In [0]:
# Import required libraries
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType
from delta.tables import DeltaTable
from pyspark.sql import Row

In [0]:
%run /Workspace/Shopvista/Shopvista-Data-Engineering-Project-_databricks/1_setup/utilities

In [0]:
print (bronze_schema, silver_schema, gold_schema)

In [0]:
dbutils.widgets.text('catalog','shopvista', 'catalog')
dbutils.widgets.text('brands_source', 'brands', 'brands_source' )
dbutils.widgets.text('category_source', 'category', 'category_source')
dbutils.widgets.text('customers_source', 'customers', 'customers_source' )
dbutils.widgets.text('date_source', 'date', 'date_source')
dbutils.widgets.text('product_source', 'products', 'product_source')
dbutils.widgets.text('db_date_source', 'db_date_table', 'db_date_source')

## CUSTOMERS

- Data transformation for business insight

In [0]:
catalog = dbutils.widgets.get('catalog')
customers_source = dbutils.widgets.get('customers_source')
print (catalog,customers_source)

In [0]:
# create a region from state column
# India states
india_region = {
    "MH": "West", "GJ": "West", "RJ": "West",
    "KA": "South", "TN": "South", "TS": "South", "AP": "South", "KL": "South",
    "UP": "North", "WB": "North", "DL": "North"
}
# Australia states
australia_region = {
    "VIC": "SouthEast", "WA": "West", "NSW": "East", "QLD": "NorthEast"
}

# United Kingdom states
uk_region = {
    "ENG": "England", "WLS": "Wales", "NIR": "Northern Ireland", "SCT": "Scotland"
}

# United States states
us_region = {
    "MA": "NorthEast", "FL": "South", "NJ": "NorthEast", "CA": "West", 
    "NY": "NorthEast", "TX": "South"
}

# UAE states
uae_region = {
    "AUH": "Abu Dhabi", "DU": "Dubai", "SHJ": "Sharjah"
}

# Singapore states
singapore_region = {
    "SG": "Singapore"
}

# Canada states
canada_region = {
    "BC": "West", "AB": "West", "ON": "East", "QC": "East", "NS": "East", "IL": "Other"
}

# Combine into a master dictionary
country_state_map = {
    "India": india_region,
    "Australia": australia_region,
    "United Kingdom": uk_region,
    "United States": us_region,
    "United Arab Emirates": uae_region,
    "Singapore": singapore_region,
    "Canada": canada_region
}  


In [0]:
# map states
country_state_map

In [0]:
# 1 Flatten country_state_map into a list of Rows
rows = []
for country, states in country_state_map.items():
    for state_code, region in states.items():
        rows.append(Row(country=country, state=state_code, region=region))
rows[:10]   

In [0]:
# 2️ Create mapping DataFrame
df_region_mapping = spark.createDataFrame(rows)

# Optional: show mapping
df_region_mapping.show(truncate=False)

In [0]:

df_silver = spark.table(f'{catalog}.{silver_schema}.{customers_source}')
#display(df_silver.limit(5))

In [0]:
df_gold = df_silver.join(df_region_mapping, on=['country', 'state'], how='left')

df_gold= df_gold.fillna({'region': 'other'})

#display(df_gold.limit(5))

In [0]:
# Write to Gold
df_gold.write\
    .format('delta')\
    .option('delta.enableChangeDataFeed', True)\
    .mode('overwrite')\
    .saveAsTable(f'{catalog}.{gold_schema}.{customers_source}')

## PRODUCTS

-- Data Transformation for business insight

In [0]:
catalog = dbutils.widgets.get('catalog')
product_source= dbutils.widgets.get('product_source')
category_source = dbutils.widgets.get('category_source')
brands_source = dbutils.widgets.get('brands_source')
print(catalog, product_source,category_source,brands_source )


In [0]:
df_brands = spark.table(f'{catalog}.{silver_schema}.{brands_source}')
df_category = spark.table(f'{catalog}.{silver_schema}.{category_source}')
df_products = spark.table(f'{catalog}.{silver_schema}.{product_source}')
                          

In [0]:
#display(df_products.limit(5))

In [0]:
#display(df_brands.limit(5))

In [0]:
#display(df_category.limit(5))

In [0]:
# Create temp views for the data_sources
df_products.createOrReplaceTempView("v_products")
df_brands.createOrReplaceTempView("v_brands")
df_category.createOrReplaceTempView("v_category")

In [0]:
#display(spark.sql("select * from v_products limit 5"))

In [0]:
#display(spark.sql('select * from v_brands limit 5'))

In [0]:
#display(spark.sql('select * from v_category limit 5'))

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{gold_schema}.{product_source} AS
WITH brands_categories AS (
  SELECT
    b.brand_name,
    b.brand_code,
    c.category_name,
    c.category_code
  FROM v_brands b
  INNER JOIN v_category c
    ON b.category_code = c.category_code
)
SELECT
  p.product_id,
  p.sku,
  p.category_code,
  COALESCE(bc.category_name, 'Not Available') AS category_name,
  p.brand_code,
  COALESCE(bc.brand_name, 'Not Available')   AS brand_name,
  p.color,
  p.size,
  p.material,
  p.weight_grams,
  p.length_cm,
  p.width_cm,
  p.height_cm,
  p.rating_count,
  p.file_name
FROM v_products p
LEFT JOIN brands_categories bc
  ON p.brand_code = bc.brand_code
""")


## DATE

In [0]:
catalog = dbutils.widgets.get('catalog')
date_source = dbutils.widgets.get('date_source')
print (catalog,date_source)

- Date Tranformation for business insight

In [0]:
df_silver = spark.table(f'{catalog}.{silver_schema}.{date_source}')
#display(df_silver.limit(5))

In [0]:
df_gold = df_silver.withColumn("date_id", F.date_format(F.col("date"), "yyyyMMdd").cast("int"))

# Add month name (e.g., 'January', 'February', etc.)
df_gold = df_gold.withColumn("month_name", F.date_format(F.col("date"), "MMMM"))

# Add is_weekend column
df_gold = df_gold.withColumn(
    "is_weekend",
    F.when(F.col("day_name").isin("Saturday", "Sunday"), 1).otherwise(0)
)

#display(df_gold.limit(5))


In [0]:
desired_columns_order = ["date_id", "date", "year", "month_name", "day_name", "is_weekend", "quarter", "week", "read_timestamp", "file_name", "file_path"]

df_gold = df_gold.select(desired_columns_order)

#display(df_gold.limit(5))

In [0]:
# write table to gold layer
df_gold.write\
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{gold_schema}.{date_source }")

## DB_DATE_TABLE

In [0]:
catalog = dbutils.widgets.get('catalog')
db_date_source = dbutils.widgets.get('db_date_source')
print(catalog, db_date_source)

In [0]:
df_silver = spark.table(f'{catalog}.{silver_schema}.{db_date_source}')
#display(df_gold.limit(6))

In [0]:
# Assign to df_silver
df_gold = df_silver
#display(df_silver.limit(5))

In [0]:
# write to silver table
df_gold.write\
    .format('delta')\
    .option('delta.enableChangeDatafeed', True)\
    .mode('overwrite')\
    .saveAsTable(f'{catalog}.{gold_schema}.{db_date_source}')
